In [12]:
import pandas as pd
import numpy as np
import scipy
import os
import kagglehub

In [6]:
def collect_from_kaggle(kaggle_path: str) -> pd.DataFrame:
    path = kagglehub.dataset_download(kaggle_path)
    file_dir = os.listdir(path)
    df_path = f"{path}/{file_dir[0]}"
    df = pd.read_csv(df_path)

    return df

In [8]:
df = collect_from_kaggle("yasserh/titanic-dataset")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
def validate_dates(df, date_col) -> tuple:
    df[date_col] = pd.datetime(df[date_col], errors = 'coerce')
    perc_error = df.isna().sum()/len(df)

    return df, perc_error

In [20]:
def get_missing_perc(df: pd.DataFrame) -> tuple:
    col_missing_perc = {}
    missing_alarm_list = []
    for col in list(df.columns):
        col_missing_perc[col] = df[col].isna().sum()/len(df)
        if df[col].isna().sum()/len(df) > 0.3:
            missing_alarm_list.append(col)

    return col_missing_perc, missing_alarm_list

missing, cols = get_missing_perc(df)

print(missing)

print(cols)
    

{'PassengerId': np.float64(0.0), 'Survived': np.float64(0.0), 'Pclass': np.float64(0.0), 'Name': np.float64(0.0), 'Sex': np.float64(0.0), 'Age': np.float64(0.19865319865319866), 'SibSp': np.float64(0.0), 'Parch': np.float64(0.0), 'Ticket': np.float64(0.0), 'Fare': np.float64(0.0), 'Cabin': np.float64(0.7710437710437711), 'Embarked': np.float64(0.002244668911335578), 'is_Age_null': np.float64(0.0), 'is_Cabin_null': np.float64(0.0)}
['Cabin']


In [ ]:
def test_mar(df: pd.DataFrame, missing_col_list: list):
    for col_target in missing_col_list:
        df[f'is_{col_target}_null'] = df[col_target].isna().astype(int)
        mar_numerical = {}
        mar_categorical = {}
        for col_other in list(df.columns):
            if col_target == col_other:
                continue
            elif df[col_other].dtype in (int, float):
                p_val = scipy.stats.pointbiserialr(df[f'is_{col_target}_null'], df[col_other]).pvalue
                if p_val < 0.05:
                    mar_numerical[(col_target, col_other)] = p_val
            else:
                crosstab = pd.crosstab(df[col_other], df[f'is_{col_target}_null'])
                p_val = scipy.stats.chi2_contingency(crosstab)[1]
                if p_val < 0.05:
                    mar_categorical[(col_target, col_other)] = p_val

    return mar_numerical, mar_categorical

mar_n, mar_c = test_mar(df, cols)

print(mar_n)
print(mar_c)

# If enough MAR checks, assume MAR and use an imputation strategy
# If not enough, if any, use an is_x_null flag


{('Cabin', 'Survived'): np.float64(3.0908910396542454e-22), ('Cabin', 'Pclass'): np.float64(1.9171279401188775e-146), ('Cabin', 'Fare'): np.float64(4.85280496497105e-53), ('Cabin', 'is_Age_null'): np.float64(1.5732394067546546e-05), ('Cabin', 'is_Cabin_null'): np.float64(0.0)}
{('Cabin', 'Sex'): np.float64(4.005446978669029e-05), ('Cabin', 'Ticket'): np.float64(5.148434664422501e-06), ('Cabin', 'Embarked'): np.float64(3.182565219539436e-11)}
